In [ ]:
# === DATA IMPORT: corrected Ge IQ files ===
from rionsis.io import load_iq_dataset

DATA_FILE = ""  # insert your filename (.npy) here
dataset = load_iq_dataset(DATA_FILE)
fs, n_samp, NF = dataset.fs, dataset.n_samples, dataset.n_files
print(f'DONE: iq {dataset.iq.shape} ({dataset.iq.nbytes/1e6:.1f} MB, {dataset.iq.dtype})')
print(f'fs={fs:.3f} Hz | duration/file={dataset.duration_s:.6f} s | {NF} files')
if all(dataset.timestamps):
    print(f'Time range: {dataset.timestamps[0]} .. {dataset.timestamps[-1]}')

In [ ]:
# === STFT AND CENTERING: generates the inputs for the staircase fit ===
from rionsis.levels import compute_power_spectrogram

DT_BIN = 0.01       # desired time step
NPERSEG = 500

spectrogram = compute_power_spectrogram(dataset.iq, fs, dt_bin=DT_BIN, nperseg=NPERSEG)
print(f'df={spectrogram.df:.2f} Hz | window={NPERSEG/fs*1e3:.3f} ms | dt(bin)={spectrogram.dt*1e3:.3f} ms')
print(f'power {spectrogram.power.shape} ({spectrogram.power.nbytes/1e6:.1f} MB, {spectrogram.power.dtype})')
print(f'Centering QC: {len(spectrogram.centering_suspect_idx)} files with |raw_gbin-gbin| > 1')

In [ ]:
# === FREQUENCY BAND CHECK ===
from rionsis.plotting import plot_frequency_bands

BAND_E   = (-2650, -1600)                      # <<< excited state (after centering)
BG_BANDS = [(-5000, -3500), (-3500, -3000)]    # <<< flat background (for the later background correction)
A1_EXTERNAL = 0.81  # only used below as a diagnostic cross-check for the calibrated A1

fig = plot_frequency_bands(spectrogram.frequency, spectrogram.power, BAND_E, BG_BANDS, n_files=NF)
fig.show()

In [ ]:
# === BAND INTEGRATION AND NOISE NORMALIZATION FOR THE STAIRCASE FIT ===
from rionsis.levels import integrate_bands, normalize_ground_noise

band_integration = integrate_bands(spectrogram.power, spectrogram.frequency, BAND_E, BG_BANDS)
noise_norm = normalize_ground_noise(band_integration.y_files, ref_sigma=0.66, noise_bins=15)
y_files_e = noise_norm.y_files
sigma_e = noise_norm.sigma
dt_e = spectrogram.dt

print(f'BAND_E={BAND_E} | y_files_e {y_files_e.shape} (files x time) | dt_e={dt_e*1e3:.3f} ms')
print(f'Noise normalization: {len(noise_norm.ground_noise_idx)} stable terminal segments '
      f'| raw sigma={noise_norm.ground_sigma_raw:.4g} -> {sigma_e}')
print(f'Ground lag-1 ACF={noise_norm.ground_acf1:+.3f}')

In [ ]:
# === INDEPENDENT A1 CALIBRATION (RUNS BEFORE THE STAIRCASE FIT) ===
from rionsis.levels import calibrate_a1
from rionsis.plotting import plot_a1_calibration

calibration = calibrate_a1(
    y_files_e, dt_e, n_max=22, mc_reps=24, em_iter=800,
    seed=20260828, sigma_hint=sigma_e, external_a1=A1_EXTERNAL,
)
A1 = calibration.a1
print('=' * 92)
print(f'A1 = {A1:.3f} +/- {calibration.sd:.3f}  (68%: {calibration.ci68[0]:.3f} .. {calibration.ci68[1]:.3f})')
print(f'By-product half-life from the mixture EM (independent cross-check, '
      f'not the primary result): {calibration.halflife_ms_byproduct:.1f} ms')
if calibration.external_a1_within_ci68 is False:
    print(f'WARNING: A1_EXTERNAL={A1_EXTERNAL} lies outside the 68% calibration band.')
print('=' * 92)

plot_a1_calibration(calibration, y_files_e, dt_e).show()

In [ ]:
# === REAL STAIRCASE FIT: physically regularized integer model ===
#
# A1 comes from the independent calibration above. The latent model
# staircase is exactly N*A1 and strictly monotonic. Bin noise is described
# by the ground-state width here; the next cell measures its actual
# dependence on N for use in the simulation generator. Real and simulated
# data go through exactly the same StaircaseFitConfig.
import numpy as np
from rionsis.levels import StaircaseFitConfig, default_change_penalty, fit_integer_dataset

FILE_START = 0
EXACT_DP_PENALTY_FACTOR = 0.30  # penalty = factor * 2*log(number of time bins)
EXACT_DP_MIN_SIZE = 1           # short physical levels remain allowed
EXACT_DP_NMAX_CAP = 80

ground_tail_bins = min(15, y_files_e.shape[1])
ground_means = y_files_e[noise_norm.ground_noise_idx, -ground_tail_bins:].mean(axis=1)
ground_level = float(np.median(ground_means))
ground_mad = 1.4826 * float(np.median(np.abs(ground_means - ground_level)))
ground_measurement_sd = sigma_e / np.sqrt(ground_tail_bins)
baseline_sd = float(np.sqrt(max(ground_mad**2 - ground_measurement_sd**2, (0.02 * A1) ** 2)))

staircase_config = StaircaseFitConfig(
    a1=A1,
    sigma2_by_n=np.full(EXACT_DP_NMAX_CAP + 1, sigma_e**2),
    tau2_by_n=np.full(EXACT_DP_NMAX_CAP + 1, baseline_sd**2),
    change_penalty=default_change_penalty(y_files_e.shape[1], EXACT_DP_PENALTY_FACTOR),
    nmax_cap=EXACT_DP_NMAX_CAP,
    min_size=EXACT_DP_MIN_SIZE,
)

fit_file_idx = np.arange(FILE_START, len(y_files_e))
real_fit = fit_integer_dataset(y_files_e[fit_file_idx], staircase_config, file_ids=fit_file_idx)
print('=' * 88)
print(f'Files={len(fit_file_idx)} | A1={A1:.4f} | sigma_bin={sigma_e:.4f} | baseline-SD={baseline_sd:.4f}')
print(f'{sum(len(v) for v in real_fit.states)} segments '
      f'({np.mean([len(v) for v in real_fit.states]):.2f} per file)')
print('=' * 88)

In [ ]:
# === NOISE MODEL sigma(N): CALIBRATED FROM REAL PLATEAUS, AND DIAGNOSTIC PLOTS ===
from rionsis.levels import calibrate_noise_vs_occupancy
from rionsis.plotting import plot_level_height_histogram, plot_noise_vs_occupancy, plot_staircase_fit

NOISE_MINLEN = 5
NOISE_MAX_N = 6

noise_calibration = calibrate_noise_vs_occupancy(
    y_files_e, real_fit.segment_lengths, real_fit.states, fit_file_idx, A1,
    min_length=NOISE_MINLEN, max_n=NOISE_MAX_N,
)
print(f'sigma(N)/A1 = sqrt({noise_calibration.sigma0:.3f}^2 + ({noise_calibration.k:.3f}*N)^2)')
plot_noise_vs_occupancy(noise_calibration).show()

PLOT_FILE_POSITIONS = (0,)
for pos in PLOT_FILE_POSITIONS:
    plot_staircase_fit(
        y_files_e[fit_file_idx[pos]], real_fit.fit_levels[pos], real_fit.segment_means[pos],
        real_fit.segment_lengths[pos], real_fit.states[pos], A1,
        title=f'File {fit_file_idx[pos]}: exact integer staircase fit',
    ).show()

real_level_heights = np.concatenate([np.asarray(v, dtype=float) for v in real_fit.segment_means])
plot_level_height_histogram(real_level_heights, A1, title='Real data: observed segment means').show()

In [ ]:
# === T1/2 HEATMAP: discrete maximum likelihood (real data) ===
from rionsis.halflife import (
    HalflifeScanConfig,
    bootstrap_halflife_grid,
    extract_terminal_events,
    scan_halflife_grid,
    summarize_bootstrap,
)
from rionsis.validation import check_geometric_mle_recovery

# Small formula test with known geometric random numbers: the estimator
# must recover a known half-life before the real-data result is trusted.
_toy_fit = check_geometric_mle_recovery(dt=dt_e)
print(f'Toy test: 37.0 ms -> {_toy_fit.halflife_ms:.2f} ms')

DISC_BOOTSTRAP = 250
DISC_BLOCK_FILES = min(8, max(1, len(fit_file_idx) // 4))
DISC_SEED = 20260827

real_events = extract_terminal_events(real_fit.segment_lengths, real_fit.states, fit_file_idx, y_files_e.shape[1])
scan_config = HalflifeScanConfig(dt=dt_e)  # default (G, L) cut grids from the original analysis
real_grid = scan_halflife_grid(real_events, scan_config)
real_boot, block_files_used = bootstrap_halflife_grid(
    real_events, fit_file_idx, scan_config, draws=DISC_BOOTSTRAP, block_files=DISC_BLOCK_FILES, seed=DISC_SEED,
)
real_boot_summary = summarize_bootstrap(real_boot)

print('=' * 96)
print(f'dt={1000*dt_e:.3f} ms | terminal 1->0 events={len(real_events)} '
      f'| bootstrap={DISC_BOOTSTRAP} x {block_files_used} files')
print('CRITICAL INTERPRETATION: L corrects only a hard detection threshold that is fully '
      'efficient above L. A stable T1/2 plateau versus L remains the decisive check; small '
      'event counts are reported in every cell.')
print('=' * 96)

In [ ]:
# === PURE DECAY SIMULATION + IDENTICAL STAIRCASE FIT ===
#
# Poisson-distributed initial ion count per file, independent exponential
# lifetimes, the STFT Hann^2 window response, and empirically resampled
# ground-state noise -- all calibrated from the real data above. Real and
# simulated data are reconstructed with the *same* staircase_config.
from rionsis.simulation import build_empirical_noise_library, hann_squared_window_response, simulate_decay_traces

SIM_T12_TRUE_MS = 37.0
SIM_NFILES = 1000  # for a more stable simulation heatmap; freely adjustable here
SIM_SEED = 20260828

noise_library = build_empirical_noise_library(y_files_e, noise_norm.ground_noise_idx, tail_bins=15)
window_response = hann_squared_window_response(NPERSEG)
real_first_bin_mean_ions = float(np.mean(y_files_e[fit_file_idx, 0]) / A1)

sim = simulate_decay_traces(
    halflife_true_ms=SIM_T12_TRUE_MS, a1=A1, dt_s=dt_e, n_time=y_files_e.shape[1], n_files=SIM_NFILES,
    nperseg=NPERSEG, fs=fs, window_response=window_response, noise_library=noise_library,
    sigma_of_occupancy=noise_calibration.sigma_of,
    initial_mean_ions=max(0.05, real_first_bin_mean_ions), seed=SIM_SEED,
)

sim_file_idx = np.arange(sim.n_files)
sim_fit = fit_integer_dataset(sim.y, staircase_config, file_ids=sim_file_idx)
print('=' * 88)
print(f'Files={SIM_NFILES} | Input T1/2={SIM_T12_TRUE_MS:.2f} ms | dt={1000*dt_e:.3f} ms')
print(f'Initial ions: target mean={max(0.05, real_first_bin_mean_ions):.3f} | realized={sim.n0.mean():.3f}')
print(f'Reconstructed staircase: {sum(len(v) for v in sim_fit.states)} segments')
print('=' * 88)

In [ ]:
# === RESULTS: T1/2 HEATMAPS AND LEVEL-HEIGHT HISTOGRAMS ===
from rionsis.plotting import plot_halflife_comparison

sim_events = extract_terminal_events(sim_fit.segment_lengths, sim_fit.states, sim_file_idx, y_files_e.shape[1])
sim_grid = scan_halflife_grid(sim_events, scan_config)
sim_boot, sim_block_files_used = bootstrap_halflife_grid(
    sim_events, sim_file_idx, scan_config, draws=DISC_BOOTSTRAP, block_files=block_files_used, seed=DISC_SEED + 1,
)
sim_boot_summary = summarize_bootstrap(sim_boot)

plot_halflife_comparison(real_grid, real_boot_summary.sd, sim_grid, sim_boot_summary.sd, SIM_T12_TRUE_MS).show()

sim_level_heights = np.concatenate([np.asarray(v, dtype=float) for v in sim_fit.segment_means])
plot_level_height_histogram(
    sim_level_heights, A1, title=f'Simulation: observed segment means ({sim.n_files} files)'
).show()

_report_l, _report_g = 5, 15
_i_l = int(np.flatnonzero(scan_config.l_values == _report_l)[0])
_i_g = int(np.flatnonzero(scan_config.g_values == _report_g)[0])
print(f'Comparison at L={_report_l}, G={_report_g}: '
      f'real={real_grid.halflife_ms[_i_g, _i_l]:.2f} ms (n={real_grid.counts[_i_g, _i_l]}), '
      f'sim={sim_grid.halflife_ms[_i_g, _i_l]:.2f} ms (n={sim_grid.counts[_i_g, _i_l]}), '
      f'simulation input={SIM_T12_TRUE_MS:.2f} ms')
print('The simulation is an end-to-end plausibility test of the staircase fit. It is not a bias '
      'correction for the real data. The decisive question is whether a common stable L range '
      'exists in the real and simulation heatmaps; NaN fields intentionally contain insufficient '
      'statistics.')